This file is used to merged the features from balance sheet, cashflow and financial data

In [5]:
import pandas as pd
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.config import MERGED_DATA

#Function to convert a statement CSV file into a DataFrame with the company, date, and metrics as a dictionary

def statement_to_dict_df(file, statement_name):
    # Extract the ticker from the filename by removing the statement name and file extension
    ticker = Path(file).stem.replace(f"_{statement_name}", "")

    df = pd.read_csv(file)

    metric_col = df.columns[0]

    rows = []

    # Iterate over each date column (starting from the second column) and create a dictionary of metrics for that date
    for date in df.columns[1:]:
        metrics = (
            df[[metric_col, date]]
            .dropna()
            .set_index(metric_col)[date]
            .to_dict()
        )

        rows.append({
            "company": ticker,
            "date": date,
            statement_name: metrics
        })

    return pd.DataFrame(rows)

Get tickers from the balance sheet folder

In [6]:
from pathlib import Path

DATA_DIR = Path("../data")
BALANCE_SHEET_DIR = DATA_DIR / "balance_sheet"
CASH_FLOW_DIR = DATA_DIR / "cashflows"
FINANCIALS_DIR = DATA_DIR / "financials"

# Get the unique tickers from the balance sheet directory
tickers = set()

for file in BALANCE_SHEET_DIR.glob("*.csv"):
    ticker = file.stem.replace("_balancesheet", "")
    tickers.add(ticker)

print(f"Found {len(tickers)} companies")

Found 1025 companies


In [7]:
master_rows = []

# Iterate over each ticker and merge the balance sheet, cash flow, and financials data into a single DataFrame
for ticker in tickers:

    try:
        bs_file = BALANCE_SHEET_DIR / f"{ticker}_balancesheet.csv"
        cf_file = CASH_FLOW_DIR / f"{ticker}_cashflow.csv"
        fin_file = FINANCIALS_DIR / f"{ticker}_financials.csv"

        bs = statement_to_dict_df(bs_file, "balancesheet")
        cf = statement_to_dict_df(cf_file, "cashflow")
        fin = statement_to_dict_df(fin_file, "financials")

        company_df = (
            bs.merge(cf, on=["company", "date"], how="outer")
              .merge(fin, on=["company", "date"], how="outer")
        )

        master_rows.append(company_df)

        print(f"Processed {ticker}")

    except Exception as e:
        # companies that are missing one of the statements will be skipped
        print(f"Failed {ticker}: {e}")


Processed MBLY
Failed SUBSZZX: 'company'
Failed MSCH: 'company'
Processed BBAI
Failed ^SGC3MILT: 'company'
Processed FIVN
Processed BETCO.ST
Processed RENT
Failed 0P0000X8PD.F: 'company'
Processed LEU
Processed WBD
Processed INTC
Processed 0M5.MU
Failed 3529.T: 'company'
Processed ATOM
Processed FFIV
Processed GH
Processed U
Failed CLICZZX: 'company'
Failed 36S.F: 'company'
Failed N26SH.NX: 'company'
Processed PEGA
Processed BEAT
Failed KARBON.BO: 'company'
Processed 7709.TWO
Processed ARLYF
Failed IKE.AX: 'company'
Processed INTA
Processed ISRG
Processed PAYO
Processed 0GJ.F
Failed 0P0001K1BA: 'company'
Processed NRIX
Failed OUTHZZX: 'company'
Processed DNA
Processed OPAL
Processed TRVG
Failed DE000SL0BQP6.SG: 'company'
Processed ARM
Processed LRCX
Failed ACQC: 'company'
Failed 0P0001I34C: 'company'
Failed CHAAZZX: 'company'
Processed AAPL
Processed EVR
Failed JAMAX: 'company'
Processed ASAI3.SA
Failed MOOVE-USD: 'company'
Failed FTCHQ: 'company'
Processed DIBS
Processed LAW
Processed

In [ ]:
import pandas as pd

dataset = pd.concat(master_rows, ignore_index=True)
dataset["date"] = pd.to_datetime(dataset["date"])
# Create a quarter column based on the date
dataset["quarter"] = dataset["date"].dt.to_period("Q").astype(str)

bs_features = pd.json_normalize(
    dataset["balancesheet"]
).add_prefix("bs_")


cf_features = pd.json_normalize(
    dataset["cashflow"]
).add_prefix("cf_")

fin_features = pd.json_normalize(
    dataset["financials"]
).add_prefix("fin_")

features_df = pd.concat(
    [
        dataset[["company", "date", "quarter"]],
        bs_features,
        cf_features,
        fin_features,
    ],
    axis=1,
)
# features_df.fillna("NaN")
# print(features_df.fillna("NaN").head())

features_df.to_csv(
    MERGED_DATA["MERGED_OUTPUT_CSV_PATH"],
    index=False
)

# Verify
print(features_df.columns)


Index(['company', 'date', 'quarter', 'bs_Cash Equivalents',
       'bs_Cash Financial', 'bs_Ordinary Shares Number', 'bs_Share Issued',
       'bs_Tangible Book Value', 'bs_Invested Capital', 'bs_Working Capital',
       ...
       'fin_Excise Taxes', 'fin_Rent Expense Supplemental',
       'fin_Rent And Landing Fees', 'fin_Other Taxes',
       'fin_Insurance And Claims', 'fin_Loss Adjustment Expense',
       'fin_Net Policyholder Benefits And Claims',
       'fin_Policyholder Benefits Gross', 'fin_Policyholder Benefits Ceded',
       'fin_Net Income From Tax Loss Carryforward'],
      dtype='str', length=342)


Compare the date from the dataset to and labelling the layoff

In [25]:
import pandas as pd


dataset = pd.read_csv(MERGED_DATA["MERGED_OUTPUT_CSV_PATH"])
layoff_df = pd.read_csv(DATA_DIR/"processed" / "layoffs_with_tickers.csv")
layoff_df["Date"] = pd.to_datetime(
    layoff_df["Date"],
    errors="coerce"
)
# Create a quarter column based on the date in the layoff_wih_tickers.csv file
layoff_df["quarter"] = layoff_df["Date"].dt.to_period("Q").astype(str)

print(dataset["quarter"].dtype)
print(dataset["quarter"].head())
# dataset["target_quarter"] = (dataset["quarter"] + 1).astype(str)

layoff_events = set(
    zip(
        layoff_df["Ticker"],
        layoff_df["quarter"]
    )
)
dataset["quarter"] = pd.PeriodIndex(
    dataset["quarter"],
    freq="Q"
)

dataset["target_quarter"] = (
    dataset["quarter"] + 1
).astype(str)


print(dataset[["quarter", "target_quarter"]].head())
layoff_events = set(
    zip(
        layoff_df["Ticker"],
        layoff_df["quarter"]
    )
)

dataset["label"] = (
    list(zip(dataset["company"], dataset["target_quarter"]))
)
dataset["label"] = dataset["label"].isin(layoff_events).astype(int)
print(dataset["label"].value_counts())


positive = dataset[dataset["label"] == 1]

print(
    positive[
        ["company", "quarter", "target_quarter"]
    ].head(20)
)
dataset.to_csv(
    MERGED_DATA["MERGED_OUTPUT_CSV_PATH"],
    index=False
)


str
0    2024Q3
1    2024Q4
2    2025Q1
3    2025Q2
4    2025Q3
Name: quarter, dtype: str
  quarter target_quarter
0  2024Q3         2024Q4
1  2024Q4         2025Q1
2  2025Q1         2025Q2
3  2025Q2         2025Q3
4  2025Q3         2025Q4
label
0    3589
1      94
Name: count, dtype: int64
    company quarter target_quarter
3      MBLY  2025Q2         2025Q3
15     FIVN  2024Q4         2025Q1
49     INTC  2025Q1         2025Q2
51     INTC  2025Q3         2025Q4
84        U  2025Q3         2025Q4
119    ISRG  2024Q3         2024Q4
129    PAYO  2025Q2         2025Q3
139   0GJ.F  2026Q1         2026Q2
430    EXPE  2025Q4         2026Q1
486     DOX  2025Q3         2025Q4
591    CRWD  2025Q2         2025Q3
651    PTON  2025Q2         2025Q3
662    SONO  2025Q1         2025Q2
665    SONO  2025Q4         2026Q1
758    GEMI  2025Q1         2025Q2
777     IBM  2024Q3         2024Q4
945    META  2025Q3         2025Q4
946    META  2025Q4         2026Q1
951    MTCH  2025Q2         2025Q3
956    C